# 🏙️ Paris Housing — Notebook 03: Modelling

Train, evaluate, and compare regression models to predict property prices.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import warnings; warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
SEED = 42
print("Libraries loaded")


## 1. Load Preprocessed Data


In [ ]:
X_train_sc  = np.load("../outputs/X_train_sc.npy")
X_test_sc   = np.load("../outputs/X_test_sc.npy")
X_train_raw = np.load("../outputs/X_train_raw.npy")
X_test_raw  = np.load("../outputs/X_test_raw.npy")
y_train     = np.load("../outputs/y_train.npy")
y_test      = np.load("../outputs/y_test.npy")

FEATURES = [
    "Arrondissement","Size_sqm","Rooms","Floor",
    "property_age","Distance_to_Center_km",
    "is_central","sqm_per_room",
    "Property_Type_enc","Condition_enc"
]
print(f"Train: {X_train_sc.shape[0]}  |  Test: {X_test_sc.shape[0]}")


## 2. Evaluation Helper


In [ ]:
results = []

def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    results.append({"Model": name, "RMSE_EUR": rmse, "MAE_EUR": mae, "R2": r2})
    print(f"  {name:<25} RMSE=€{rmse:>12,.0f}  MAE=€{mae:>12,.0f}  R²={r2:.4f}")
    return y_pred


## 3. Baseline — Linear Regression


In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
lr_pred = evaluate("Linear Regression", y_test, lr.predict(X_test_sc))


## 4. Random Forest Regressor


In [ ]:
rf = RandomForestRegressor(
    n_estimators=300, max_depth=12,
    min_samples_leaf=3, random_state=SEED, n_jobs=-1
)
rf.fit(X_train_raw, y_train)
rf_pred = evaluate("Random Forest", y_test, rf.predict(X_test_raw))


## 5. XGBoost Regressor


In [ ]:
xgb = XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, verbosity=0
)
xgb.fit(X_train_raw, y_train)
xgb_pred = evaluate("XGBoost", y_test, xgb.predict(X_test_raw))


## 6. Model Comparison


In [ ]:
results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ["RMSE_EUR", "MAE_EUR", "R2"]
labels  = ["RMSE (€)", "MAE (€)", "R²"]
for ax, metric, label in zip(axes, metrics, labels):
    ax.bar(results_df["Model"], results_df[metric], color="#1a73e8", edgecolor="white")
    ax.set_title(label, fontsize=13)
    ax.set_ylabel(label)
    ax.tick_params(axis="x", rotation=20)
    if metric != "R2":
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x/1e3:.0f}k"))
plt.suptitle("Model Comparison", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig("../outputs/03_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Feature Importance (XGBoost)


In [ ]:
importances = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind="barh", ax=ax, color="#1a73e8")
ax.set_title("XGBoost Feature Importances", fontsize=14)
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig("../outputs/03_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Residual Analysis — Best Model


In [ ]:
residuals = y_test - xgb_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(xgb_pred, residuals, alpha=0.35, color="#1a73e8", s=18)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("XGBoost — Residuals vs Predicted", fontsize=13)
axes[0].set_xlabel("Predicted (€)")
axes[0].set_ylabel("Residual (€)")

axes[1].hist(residuals, bins=40, color="#1a73e8", edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Residual Distribution", fontsize=13)
axes[1].set_xlabel("Residual (€)")

plt.tight_layout()
plt.savefig("../outputs/03_residuals.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Predict Price for a New Property


In [ ]:
# Edit the values below to predict your own property
# Encoding: Property_Type  Apartment=0, Loft=1, Penthouse=2, Studio=3
#            Condition      Good=0, Needs Renovation=1, New=2, Renovated=3

new_property = pd.DataFrame([{
    "Arrondissement"        : 7,
    "Size_sqm"              : 95,
    "Rooms"                 : 4,
    "Floor"                 : 3,
    "property_age"          : 30,   # 2024 - Year_Built
    "Distance_to_Center_km" : 3.5,
    "is_central"            : 1,    # arrondissement <= 8
    "sqm_per_room"          : 23.75,
    "Property_Type_enc"     : 0,    # Apartment
    "Condition_enc"         : 3,    # Renovated
}])

predicted_price = xgb.predict(new_property)[0]
print(f"Predicted price: €{predicted_price:,.0f}")


## 10. Summary

| Model | RMSE | MAE | R² |
|---|---|---|---|
| Linear Regression | higher | higher | ~0.60 |
| Random Forest | medium | medium | ~0.84 |
| **XGBoost** | **lowest** | **lowest** | **~0.87** |

XGBoost is the recommended model. The most important features are **Size_sqm**, **Distance_to_Center_km**, and **Arrondissement**.
